# Flujo de validación y generación

Notebook para recorrer paso a paso el flujo de la app:
    print('- Comparacion rapida con Certificacion -')
- Validar ER y ESF (determinístico)
- Validar Cédula/Matrícula (OCR + reglas)
- (Opcional) Validación LLM
- Generar DOCX y guardar reporte JSON

Rellena las rutas en la celda de configuración y ejecuta las celdas en orden.

In [2]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Configuración básica
EXCEL_PATH     = Path(r'C:\Users\yamil\OneDrive\Certificaciones\FICOSHA\Efraín Alejandro Hernández Sequeira\Efraín Alejandro Hernández Sequeira.xlsx')
OUTPUT_DIR     = Path(r'C:\Users\yamil\OneDrive\Certificaciones\FICOSHA\Efraín Alejandro Hernández Sequeira')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DOCX    = OUTPUT_DIR / 'Certificacion_Efrain_Alejandro_Hernandez_Sequeira.docx'

# Si dejas vacío, el cuaderno intentará buscar en input_docs/...
CEDULA_PATH      = Path(r'')
CEDULA_BACK_PATH = Path(r'')

TOLERANCIA      = 1.0       # o 5.0/10.0 si hay redondeos mayores
STRICT_CONTABLE = False     # True para detener si ER/ESF no cuadran
STRICT_DOCS     = False     # True para exigir cédula/matrícula

RUN_LLM   = True
LLM_MODEL = 'gpt-4o'

def assert_paths():
    assert EXCEL_PATH.exists(), f'No existe el Excel: {EXCEL_PATH}'
    if CEDULA_PATH:
        assert CEDULA_PATH.exists(), f'No existe la cédula: {CEDULA_PATH}'
    if CEDULA_BACK_PATH and str(CEDULA_BACK_PATH):
        assert CEDULA_BACK_PATH.exists(), f'No existe el reverso de cédula: {CEDULA_BACK_PATH}'

# Autocompletar rutas desde carpetas estándar si están vacías
from path_utils import find_first_file
def _is_file(p):
    try:
        return p and str(p) and Path(p).exists() and Path(p).is_file()
    except Exception:
        return False
if not _is_file(CEDULA_PATH):
    f = find_first_file('input_docs/cedula/front')
    if f: CEDULA_PATH = Path(f)
if not _is_file(CEDULA_BACK_PATH):
    b = find_first_file('input_docs/cedula/back')
    if b: CEDULA_BACK_PATH = Path(b)
print('Auto CEDULA_PATH=', CEDULA_PATH)
print('Auto CEDULA_BACK_PATH=', CEDULA_BACK_PATH)
assert_paths()


Auto CEDULA_PATH= d:\Proyectos\certificacion_app\input_docs\cedula\front\cedula.jpeg
Auto CEDULA_BACK_PATH= d:\Proyectos\certificacion_app\input_docs\cedula\back\cedula_reverso.jpeg


In [3]:
# Imports y preparación de entorno
import os, sys, json
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

# Asegurar que se puedan importar los módulos del repo
repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from excel_reader import ExcelData
from generators.utils import extract_cert_fields
from validators import validate_er, validate_esf
import importlib, llm_validation as _llmv
_ = importlib.reload(_llmv)
from llm_validation import build_snapshot, llm_validate
llm_extract_cedula_from_text = getattr(_llmv, 'llm_extract_cedula_from_text', None)
from document_generator import generar_documento_completo
from report_utils import build_report, save_report_json

print('Entorno listo. OPENAI_API_KEY presente? ', bool(os.getenv('OPENAI_API_KEY')))



Entorno listo. OPENAI_API_KEY presente?  True


d:\Proyectos\certificacion_app\.venv\Lib\site-packages\docxcompose\properties.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Cargar Excel y mostrar vista rápida
assert EXCEL_PATH, 'Define EXCEL_PATH'
data = ExcelData(EXCEL_PATH)
df_esf   = data.get_situacion_financiera()
df_er    = data.get_resultados()
df_datos = data.get_datos()
df_cert  = data.get_certificacion()

print('ESF:', df_esf.shape, 'ER:', df_er.shape, 'Datos:', df_datos.shape, 'Cert:', df_cert.shape)
display(df_cert.head(10))
display(df_er.head(8))
display(df_esf.head(8))


ESF: (22, 7) ER: (26, 10) Datos: (10, 3) Cert: (23, 3)


d:\Proyectos\certificacion_app\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Descripción,Datos,Check List
0,Nombre completo,Efraín Alejandro Hernández Sequeira,NaN
1,Cedula,201-251184-0000Q,0.0
2,Fecha Inicio,2025-04-01 00:00:00,0.0
3,Fecha Fin,2025-09-30 00:00:00,0.0
4,Estado Civil,en unión de hecho estable,0.0
5,Profesion,Agronomo,0.0
6,Sexo,Masculino,0.0
7,Domicilio,"Managua, Departamento de Managua",0.0
8,Dirección Personal,"Barrio México, Autolote El Chele, 60 varas al ...",0.0
9,Dirección Negocio,"Barrio Concepción de María, kilómetro 9.5 Carr...",0.0


,Descripción,Dólares,2025-04-30 00:00:00,2025-05-31 00:00:00,2025-06-30 00:00:00,2025-07-31 00:00:00,2025-08-31 00:00:00,2025-09-30 00:00:00,Acumulado de 6 meses,Promedio Mensual
0,NaN,NaN,0.9491,0.9918,0.9105,1.1327,0.9490,1.0491,NaN,NaN
1,NaN,NaN,0.4252,0.4615,0.4466,0.4443,0.4760,0.4655,NaN,NaN
2,NaN,NaN,36.6243,36.6243,36.6243,36.6243,36.6243,36.6243,NaN,NaN
3,1,NaN,834243.0000,871776.0000,800314.0000,995624.0000,834155.0000,922141.0000,NaN,NaN
4,0,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Ingresos,24000.0,834243.0000,871776.0000,800314.0000,995624.0000,834155.0000,922141.0000,5258253.0,876376.0
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Activos,Unnamed: 1,Unnamed: 2,Pasivos,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Corrientes,NaN,NaN,Corrientes,NaN,NaN,NaN
1,Efectivo y Equivalentes de Efectivo,311307.0,NaN,Proveedores,0.0,NaN,NaN
2,Cuentas por Cobrar,0.0,NaN,Tarjetas de Crédito,32962.0,NaN,NaN
3,Inventarios,2820071.0,NaN,NaN,NaN,NaN,NaN
4,Total Corrientes,3131378.0,NaN,Total Corrientes,32962.0,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,No Corrientes,NaN,NaN,No Corrientes,NaN,NaN,NaN
7,Propiedad Planta y Equipos,NaN,NaN,Créditos Prendarios,805735.0,NaN,NaN


In [5]:
# Extraer campos de 'Certificacion' (robusto)
cert = extract_cert_fields(df_cert)
cert


{'nombre': None,
 'apellido': None,
 'nombre_completo': 'Efraín Alejandro Hernández Sequeira',
 'cedula': '201-251184-0000Q',
 'inicio': Timestamp('2025-04-01 00:00:00'),
 'fin': Timestamp('2025-09-30 00:00:00'),
 'estado_civil': 'en unión de hecho estable',
 'profesion': 'Agronomo',
 'sexo': 'Masculino',
 'domicilio': 'Managua, Departamento de Managua',
 'direccion_personal': 'Barrio México, Autolote El Chele, 60 varas al este, casa N.º 11, Municipio de Managua, Departamento de Managua.',
 'direccion_negocio': 'Barrio Concepción de María, kilómetro 9.5 Carretera Norte, frente a los semáforos de la Subasta, Municipio de Managua, Departamento de Managua.',
 'primer_apellido': 'Hernández',
 'ingresos_brutos': 5258253.0,
 'ingresos_promedio': 876376.0,
 'utilidad_periodo': 2666897.0,
 'utilidad_promedio': 444483.0,
 'banco': 'FICOHSA',
 'fecha_certificacion': Timestamp('2025-10-17 00:00:00')}

In [6]:
import pandas as pd

def debug_er(df_er):
    df = (df_er.copy()
           .drop(index=[0,1,2,3,4])
           .reset_index(drop=True)
           .drop(columns=[df_er.columns[1]])
           .fillna(0))
    desc = df.columns[0]

    def find(opts):
        opts = {str(x).strip().lower() for x in opts}
        for i,row in df.iterrows():
            if str(row[desc]).strip().lower() in opts:
                return i
        return None

    i_ing  = find(['(=) ingresos brutos','ingresos brutos','ingresos'])
    i_gop0 = find(['(-) gastos operativos','gastos operativos'])
    i_gopT = find(['total gastos operativos'])
    i_util = find(['utilidad operativa','ingresos/utilidad neta','utilidad neta','resultado neto'])

    out=[]
    for c in df.columns[1:]:
        ing  = pd.to_numeric(df.loc[i_ing, c], errors='coerce')
        gopT = pd.to_numeric(df.loc[i_gopT, c], errors='coerce')
        ureal= pd.to_numeric(df.loc[i_util, c], errors='coerce')
        ucalc    = ing - gopT
        ucalcAbs = ing - abs(gopT)
        out.append({
            'col': str(c),
            'ingresos': float(ing or 0),
            'gop_total': float(gopT or 0),
            'util_real': float(ureal or 0),
            'calc': float(ucalc or 0),
            'calc_abs': float(ucalcAbs or 0),
            'diff': float((ucalc - ureal) if pd.notna(ucalc) and pd.notna(ureal) else 0),
            'diff_abs': float((ucalcAbs - ureal) if pd.notna(ucalcAbs) and pd.notna(ureal) else 0),
        })
    return pd.DataFrame(out)

dbg = debug_er(df_er)
dbg


,col,ingresos,gop_total,util_real,calc,calc_abs,diff,diff_abs
0,2025-04-30 00:00:00,834243.0,34482.0,445041.0,799761.0,799761.0,354720.0,354720.0
1,2025-05-31 00:00:00,871776.0,34778.0,434673.0,836998.0,836998.0,402325.0,402325.0
2,2025-06-30 00:00:00,800314.0,33983.0,408911.0,766331.0,766331.0,357420.0,357420.0
3,2025-07-31 00:00:00,995624.0,35856.0,517412.0,959768.0,959768.0,442356.0,442356.0
4,2025-08-31 00:00:00,834155.0,34161.0,402936.0,799994.0,799994.0,397058.0,397058.0
5,2025-09-30 00:00:00,922141.0,34960.0,457924.0,887181.0,887181.0,429257.0,429257.0
6,Acumulado de 6 meses,5258253.0,208220.0,2666897.0,5050033.0,5050033.0,2383136.0,2383136.0
7,Promedio Mensual,876376.0,34704.0,444483.0,841672.0,841672.0,397189.0,397189.0


In [7]:
# Validación ER y ESF (determinística)
v_er  = validate_er(df_er,  tolerance=TOLERANCIA)
v_esf = validate_esf(df_esf, tolerance=TOLERANCIA)

def resumen_val(v):
    return dict(ok=v.get('ok'), n_errors=len(v.get('errors', [])), n_checks=len(v.get('checks', [])))

print('ER ->', resumen_val(v_er))
print('ESF->', resumen_val(v_esf))

# Mostrar checks fallidos
fails_er  = [c for c in v_er.get('checks', []) if not c.get('ok', True)]
fails_esf = [c for c in v_esf.get('checks', []) if not c.get('ok', True)]
print('ER fallos:', len(fails_er))
for c in fails_er[:10]:
    print('-', c)
print('ESF fallos:', len(fails_esf))
for c in fails_esf[:10]:
    print('-', c)


ER -> {'ok': True, 'n_errors': 0, 'n_checks': 16}
ESF-> {'ok': True, 'n_errors': 0, 'n_checks': 6}
ER fallos: 0
ESF fallos: 0


In [8]:
res_esf = validate_esf(df_esf, tolerance=TOLERANCIA)
for chk in res_esf["checks"]:
    print(chk)


{'rule': 'ESF: Total Corrientes = suma detalle', 'side': 'izq', 'expected': 3131378.0, 'computed': 3131378.0, 'ok': True}
{'rule': 'ESF: Total No Corrientes = suma detalle', 'side': 'izq', 'expected': 1323054.0, 'computed': 1323054.0, 'ok': True}
{'rule': 'ESF: Total Activos = Total Corrientes + Total No Corrientes', 'expected': 4454432.0, 'computed': 4454432.0, 'ok': True}
{'rule': 'ESF: Total Patrimonio = suma detalle', 'side': 'der', 'expected': 3615735.0, 'computed': 3615735.0, 'ok': True}
{'rule': 'ESF: Total Pasivo + Patrimonio = Total Pasivos + Total Patrimonio', 'expected': 4454432.0, 'computed': 4454432.0, 'ok': True}
{'rule': 'ESF: Total Activos = Total Pasivo + Patrimonio', 'expected': 4454432.0, 'computed': 4454432.0, 'ok': True}


## === LLM_VISION_EXTRACT (Cédula) ===

Usa un modelo con visión para extraer JSON directamente desde la imagen del frente (y reverso si está).

## LLM Visión: extracción paso a paso

Ejecuta la detección con GPT-4o-mini sobre frente y reverso, muestra los campos y normaliza la cédula y el nombre.

In [8]:
# Import robusto + autodetección de rutas
try:
    from llm_vision import extract_cedula_with_vision
except ModuleNotFoundError:
    import os, sys
    from pathlib import Path
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    os.chdir(ROOT)
    from llm_vision import extract_cedula_with_vision

import json, re, unicodedata
LLM_MODEL = globals().get('LLM_MODEL', 'gpt-4o-mini')
front = str(globals().get('CEDULA_PATH', '') or '')
back  = str(globals().get('CEDULA_BACK_PATH', '') or '') or None
if not front:
    try:
        from path_utils import find_first_file
    except ModuleNotFoundError:
        from pathlib import Path as _Path
        import sys as _sys, os as _os
        _ROOT = _Path.cwd()
        if _ROOT.name == 'notebooks':
            _ROOT = _ROOT.parent
        if str(_ROOT) not in _sys.path:
            _sys.path.insert(0, str(_ROOT))
        _os.chdir(_ROOT)
        from path_utils import find_first_file
    front = find_first_file('input_docs/cedula/front') or ''
    back = back or find_first_file('input_docs/cedula/back')
assert front, 'Falta imagen de frente de cédula'
print('Front:', front)
print('Back :', back)
vision = extract_cedula_with_vision(front, back, model=LLM_MODEL)
print(json.dumps(vision, ensure_ascii=False, indent=2))


Front: d:\Proyectos\certificacion_app\input_docs\cedula\front\cedula.jpeg
Back : d:\Proyectos\certificacion_app\input_docs\cedula\back\cedula_reverso.jpeg
{
  "ok": true,
  "fields": {
    "cedula": "201-251184-0000Q",
    "nombres": "EFRAIN ALEJANDRO",
    "apellidos": "HERNANDEZ SEQUEIRA",
    "nombre_completo": "EFRAIN ALEJANDRO HERNANDEZ SEQUEIRA",
    "nacimiento": "25-11-1984",
    "sexo": "M",
    "emision": "27-05-2024",
    "expiracion": "27-05-2034",
    "lugar_nacimiento": "GRANADA"
  }
}


In [9]:
import re
import unicodedata

def _norm(s):
    s = unicodedata.normalize('NFD', str(s or ''))
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
    return ' '.join(s.upper().split())

def _norm_ced(ced):
    """
    Normaliza a ###-######-####L (L opcional). Si no calza, retorna la entrada limpia.
    """
    t = re.sub(r'[^0-9A-Za-z]', '', str(ced or ''))
    m = re.match(r'(\d{3})(\d{6})(\d{4})([A-Za-z]?)$', t)
    if m:
        letra = m.group(4).upper()
        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}{letra}"
    return t  # podrías devolver '' si prefieres forzar formato estricto

# Fuente de campos (si vision no existe, caemos a dict vacío)
fields = {}
try:
    fields = (vision or {}).get('fields', {})
except NameError:
    fields = {}

print('- Campos normalizados -')
print('Cedula:', _norm_ced(fields.get('cedula')))
print('Nombres:', _norm(fields.get('nombres')))
print('Apellidos:', _norm(fields.get('apellidos')))
print('Nombre completo:', _norm(fields.get('nombre_completo')))
print('Nacimiento:', fields.get('nacimiento'))
print('Sexo:', fields.get('sexo'))
print('Emision:', fields.get('emision'))
print('Expiracion:', fields.get('expiracion'))
print('Lugar de nacimiento:', _norm(fields.get('lugar_nacimiento')))

# Comparación con hoja Certificación (si df_cert existe)
try:
    from generators.utils import extract_cert_fields
    from vision_validation import _check_name, _check

    cert = extract_cert_fields(df_cert)  # df_cert debe existir en el entorno
    exp_full = (cert.get('nombre_completo')
                or f"{cert.get('nombre') or ''} {cert.get('apellido') or ''}").strip()

    got_full = (fields.get('nombre_completo')
                or f"{fields.get('nombres') or ''} {fields.get('apellidos') or ''}").strip()

    checks = []
    checks.append({'doc': 'cedula_vision', **_check(cert.get('cedula'), fields.get('cedula'), label='cedula')})
    # Si tu _check_name espera (esperado, esperado_apellidos, obtenido, policy),
    # ajusta el segundo parámetro según tu contrato. Aquí dejamos '' como placeholder.
    checks.append({'doc': 'cedula_vision', **_check_name(exp_full, '', got_full, policy='balanced')})

    v_cedula_vision = {
        'ok': all(c.get('ok', False) for c in checks),
        'checks': checks,
        'fields': fields
    }

    print('\n- Vision vs Certificacion -')
    for c in checks:
        print(c)

except NameError as e:
    # df_cert o los módulos podrían no existir
    print('No fue posible comparar con df_cert usando LLM Vision (NameError):', e)
except ImportError as e:
    print('No fue posible importar utilidades de validación:', e)
except Exception as e:
    print('No fue posible comparar con df_cert usando LLM Vision:', e)


- Campos normalizados -
Cedula: 201-251184-0000Q
Nombres: EFRAIN ALEJANDRO
Apellidos: HERNANDEZ SEQUEIRA
Nombre completo: EFRAIN ALEJANDRO HERNANDEZ SEQUEIRA
Nacimiento: 25-11-1984
Sexo: M
Emision: 27-05-2024
Expiracion: 27-05-2034
Lugar de nacimiento: GRANADA

- Vision vs Certificacion -
{'doc': 'cedula_vision', 'field': 'cedula', 'ok': True, 'expected': '201-251184-0000Q', 'got': '201-251184-0000Q'}
{'doc': 'cedula_vision', 'field': 'nombre', 'ok': True, 'expected': 'Efraín Alejandro Hernández Sequeira', 'got': 'EFRAIN ALEJANDRO HERNANDEZ SEQUEIRA', 'coverage': {'full': 1.0}, 'policy': 'balanced'}


In [10]:
# LLM visión (mínimo) con import y rutas robustas
try:
    from llm_vision import extract_cedula_with_vision
except ModuleNotFoundError:
    import os, sys
    from pathlib import Path
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    os.chdir(ROOT)
    from llm_vision import extract_cedula_with_vision

fp = str(globals().get('CEDULA_PATH','') or '')
bp = str(globals().get('CEDULA_BACK_PATH','') or '') or None
if not fp:
    try:
        from path_utils import find_first_file
    except ModuleNotFoundError:
        from pathlib import Path as _Path
        import sys as _sys, os as _os
        _ROOT = _Path.cwd()
        if _ROOT.name == 'notebooks': _ROOT = _ROOT.parent
        if str(_ROOT) not in _sys.path: _sys.path.insert(0, str(_ROOT))
        _os.chdir(_ROOT)
        from path_utils import find_first_file
    fp = find_first_file('input_docs/cedula/front') or ''
    bp = bp or find_first_file('input_docs/cedula/back')
if not fp:
    print('No hay imagen de frente para LLM visión')
else:
    vis = extract_cedula_with_vision(fp, bp, model=globals().get('LLM_MODEL','gpt-4o-mini'))
    print(vis)


{'ok': True, 'fields': {'cedula': '201-251184-0000Q', 'nombres': 'EFRAIN ALEJANDRO', 'apellidos': 'HERNANDEZ SEQUEIRA', 'nombre_completo': 'HERNANDEZ SEQUEIRA EFRAIN ALEJANDRO', 'nacimiento': '25-11-1984', 'sexo': 'M', 'emision': '27-05-2024', 'expiracion': '27-05-2034', 'lugar_nacimiento': 'GRANADA'}}


In [11]:
# Validación LLM (opcional)
v_llm = None

# Preferir resultados de LLM Visión como 'documentos'
doc_checks = None
try:
    doc_checks = v_cedula_vision if ('v_cedula_vision' in globals() and v_cedula_vision) else None
except NameError:
    doc_checks = None

# Compatibilidad: si existe v_docs previo, combinar
if doc_checks is None:
    try:
        doc_checks  # noqa: F401
    except NameError:
        try:
            v_docs
        except NameError:
            v_docs = {'ok': True, 'checks': []}
        doc_checks = v_docs

if RUN_LLM:
    snap = build_snapshot(df_er, df_esf, df_cert, v_er, v_esf, doc_checks)
    v_llm = llm_validate(snap, model=LLM_MODEL)
    print('LLM ok:', v_llm.get('ok'), '| issues:', len(v_llm.get('issues', [])))
    print('Resumen:', v_llm.get('summary', ''))
else:
    print('LLM desactivado.')

# Detalle de issues reportados por el LLM
try:
    issues = v_llm.get('issues', []) if isinstance(v_llm, dict) else []
    if issues:
        print('\n== LLM - Detalle de issues ==')
        from collections import Counter
        sev_cnt = Counter([(it.get('severity','') or '').lower() for it in issues])
        if sev_cnt:
            print('Conteo por severidad:', dict(sev_cnt))
        for idx, it in enumerate(issues, 1):
            sev = it.get('severity','')
            desc = it.get('description','')
            ev  = it.get('evidence','')
            sug = it.get('suggestion','')
            print(str(idx)+'. ['+str(sev)+'] '+str(desc))
            if ev: print('   Evidencia:', ev)
            if sug: print('   Sugerencia:', sug)
    else:
        print('\n(Sin issues reportados por el LLM)')
except Exception as e:
    print('No fue posible listar issues LLM:', e)


LLM ok: True | issues: 0
Resumen: La validación de los datos financieros y documentales es satisfactoria.

(Sin issues reportados por el LLM)


In [12]:
# Generar el documento (con páginas de validación)
if OUTPUT_DOCX:
    try:
        generar_documento_completo(
            df_esf=df_esf,
            df_er=df_er,
            df_datos=df_datos,
            df_cert=df_cert,
            output_path=OUTPUT_DOCX,
            incluir_validacion=True,
            tolerancia_validacion=TOLERANCIA,
            detener_si_error=STRICT_CONTABLE,
            validacion_documentos=v_docs,
            validacion_llm=v_llm,
        )
        print('DOCX generado:', OUTPUT_DOCX)
    except Exception as e:
        print('Error al generar DOCX:', type(e).__name__, e)
else:
    print('Define OUTPUT_DOCX si deseas generar el documento.')


Error al generar DOCX: NameError name 'v_docs' is not defined


In [6]:
# Guardar reporte JSON con todos los resultados
report = build_report(v_er=v_er, v_esf=v_esf, v_docs=v_docs, v_llm=v_llm, meta={'excel': EXCEL_PATH, 'output': OUTPUT_DOCX})
if OUTPUT_DOCX:
    path_json = save_report_json(report, OUTPUT_DOCX)
else:
    # si no hay DOCX, guarda junto al Excel con sufijo .validation.json
    base, _ = os.path.splitext(EXCEL_PATH)
    path_json = base + '.validation.json'
    with open(path_json, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
print('Reporte guardado en:', path_json)


NameError: name 'v_er' is not defined

## LLM Visión: Matrícula (Registro Contable)

Extrae campos (ROC, dirección, RUC, nombre) desde la imagen y compara con la hoja Certificación.

In [9]:
import importlib, vision_validation as _vv
_vv = importlib.reload(_vv)
from vision_validation import validate_matricula_vision
from llm_vision import extract_matricula_with_vision

from vision_validation import validate_matricula_vision
from llm_vision import extract_matricula_with_vision

# Intentar localizar automáticamente la imagen si no está definida
mat_path = ''
try:
    from path_utils import find_first_file
    mat_path = find_first_file('input_docs/matricula') or ''
except Exception:
    mat_path = ''

if not mat_path:
    print('No hay imagen de matrícula en input_docs/matricula')
else:
    print('Matrícula:', mat_path)
    mv = validate_matricula_vision(df_cert, matricula_path=mat_path, model=LLM_MODEL)
    print('Matrícula ok:', mv.get('ok', False), '| fallos:', sum(1 for c in mv.get('checks', []) if not c.get('ok', False)))
    for c in mv.get('checks', []):
        print('-', c)
    # También puedes inspeccionar la extracción cruda
    raw = extract_matricula_with_vision(mat_path, model=LLM_MODEL)
    print('Extraído (raw):', raw.get('fields', {}))


Matrícula: d:\Proyectos\certificacion_app\input_docs/matricula\matricula 2025.pdf
Matrícula ok: False | fallos: 2
- {'doc': 'matricula_vision', 'field': 'matricula', 'ok': False, 'error': 'No encontrado'}
- {'doc': 'matricula_vision', 'field': 'direccion', 'ok': False, 'error': 'No encontrado'}
Extraído (raw): {'roc': None, 'direccion': None, 'ruc': None, 'nombre': None, 'fecha_emision': None}
